# Implementation: physics loss, Greenshields vs Underwood, time-aligned loss, METR-LA check (CS637)

This notebook reproduces everything in `implementation/`:

| Section | What | Runtime | Time |
|---|---|---|---|
| 1 | Setup: this repo, the authors' variant folders, PEMSD8 and METR-LA data | any | ~5 min |
| 2 | Tests: our physics loss equals the authors' | CPU is enough | ~1 min |
| 3 | Per-sensor Underwood and Greenshields fits (training split) | CPU is enough | ~1 min |
| 4 | Training runs (13 runs, see table in section 4) | T4 GPU | ~55 min per PEMSD8 run, ~70 min per METR-LA run |
| 5 | Evaluation per horizon (automatic after each run, or by hand) | any | ~15 s (PEMSD8), ~45 s (METR-LA) per run |
| 6 | Summaries and figures (3-seed comparisons) | CPU | ~1 min |
| 7 | Calibration check of the time-aligned loss (conformal) | CPU is enough | ~20 min |

The results of our runs are already in `results/`, `conformal/`, `figures/` and `logs/`; re-running is only needed to reproduce them.

Notes on free Colab:
- Sessions can end after 35 min to 2 h. Runs therefore train on the VM's local disk, a background loop copies checkpoints and logs to Drive every 5 minutes, and each run is evaluated automatically when training ends.
- Give each training run a GPU of its own: two PEMSD8 runs sharing one T4 took 2.9 min per epoch each, against 1.0-1.1 min alone.
- Switching the runtime type wipes `/content`; run section 1 again afterwards.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, json, shutil, subprocess, requests
import numpy as np, torch
print('GPU:', torch.cuda.is_available())

WORK  = '/content/drive/MyDrive/CS637/implementation_work'   # results, logs and checkpoint backups (Drive)
LOCAL = '/content/runs'                                      # run folders (local disk: faster, and Drive copies of many small files fail)
REPO  = '/content/cs637'                                     # this repository
AUTH  = '/content/authors'                                   # the authors' repository (only the folders we need)
for d in (WORK, f'{WORK}/results', f'{WORK}/logs', f'{WORK}/params', LOCAL):
    os.makedirs(d, exist_ok=True)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/Vasu-63/cs637-uq-traffic-evaluation', REPO], check=True)
IMPL = f'{REPO}/implementation'
METRICS_SCRIPTS = f'{REPO}/reproduction/scripts'             # horizon_metrics.py, used by eval_run.py

VARIANTS = ['gpde_quantile_phy_model_PEMSD8',   # PEMSD8 GPDE+Quantile+Phy, lambda 0.002 (our runs)
            'gpde_quantile_model',              # METR-LA GPDE+Quantile
            'gpde_quantile_phy_model']          # METR-LA GPDE+Quantile+Phy, lambda 0.0005
if not os.path.isdir(AUTH):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--no-checkout',
                    'https://github.com/tianshubao1/uq4phy_traffic_network', AUTH], check=True)
    subprocess.run(['git', '-C', AUTH, 'checkout', 'HEAD', '--'] + VARIANTS, check=True)
for p in glob.glob(f'{AUTH}/**/__pycache__', recursive=True):
    shutil.rmtree(p)

def get_figshare(article_id, name):
    out = f'/content/data/{name}'
    if os.path.isdir(out):
        return
    os.makedirs(out)
    for f in requests.get(f'https://api.figshare.com/v2/articles/{article_id}/files').json():
        with requests.get(f['download_url'], stream=True) as r, open(os.path.join(out, f['name']), 'wb') as fh:
            for c in r.iter_content(1 << 20):
                fh.write(c)
    print(name, sorted(os.listdir(out)))

get_figshare(28347662, 'PEMSD8')    # train/val/test.npz + pems8ndy.npy
get_figshare(28347830, 'METR-LA')

## 2. Tests: our physics loss equals the authors'

In [ ]:
!pip -q install pytest
!cd {REPO} && AUTHORS_DIR={AUTH}/gpde_quantile_phy_model_PEMSD8 python -m pytest -q implementation/tests

## 3. Per-sensor fits of both speed-density laws (PEMSD8 training split)

Writes `underwood_*.npy`, `greenshields_*.npy` (170 values each) and `fit_quality.csv`. The Underwood fit reproduces the authors' shipped `PEMSD8_v_f.npy` / `PEMSD8_rho_max.npy`.

In [ ]:
!python {IMPL}/fit_params.py --data /content/data/PEMSD8/pems8ndy.npy --out {WORK}/params
PARAMS = f'{WORK}/params'
print(sorted(os.listdir(PARAMS)))

## 4. Training runs

| Run label (as in `results/horizon_metrics.csv`) | Base folder | Change | Seeds |
|---|---|---|---|
| `pemsd8_underwood_ours`, `_seed1`, `_seed2` | PEMSD8 GPDE+Quantile+Phy | our `physics_losses.py` instead of the authors' loss (import line only) | 0, 1, 2 |
| `pemsd8_greenshields`, `_seed1`, `_seed2` | same | `PHYS_LAW=greenshields` + Greenshields parameters | 0, 1, 2 |
| `pemsd8_underwood_aligned`, `_seed1`, `_seed2` | same | time-aligned physics loss: `PHYS_ALIGN=future` (true future occupancy from the labels) | 0, 1, 2 |
| `metrla_gpde_quantile`, `_seed1` | METR-LA GPDE+Quantile | none (25 epochs) | 0, 1 |
| `metrla_gpde_quantile_phy`, `_seed1` | METR-LA GPDE+Quantile+Phy | none (25 epochs) | 0, 1 |

All runs: batch 128, learning rate 0.01, PEMSD8 50 epochs, METR-LA 25 epochs (the authors' configs say 50). Seed 0 is the authors' hard-coded `random_seed = 0`; for seeds 1 and 2 only that line changes, so the same seed gives the same initialisation for every loss and comparisons are paired.

The next cell prepares the base folders and the helpers. Then start runs with `start([...])`, one GPU per run (examples below).

In [ ]:
BASES = {}

def prepare_base(variant, name, data_dir, epochs=None, our_loss=False):
    dst = f'{LOCAL}/_base_{name}'
    if not os.path.isdir(dst):
        shutil.copytree(f'{AUTH}/{variant}', dst, ignore=shutil.ignore_patterns('models', 'info.log', '__pycache__'))
        c = f'{dst}/data/model/dcrnn_la.yaml'
        t = re.sub(r'dataset_dir:.*', f'dataset_dir: {data_dir}', open(c).read())
        if epochs:
            t = re.sub(r'(\bepochs:\s*)\d+', rf'\g<1>{epochs}', t)
        open(c, 'w').write(t)
        if our_loss:   # swap the authors' underwood_loss for ours: copy the module, change one import
            shutil.copy(f'{IMPL}/physics_losses.py', f'{dst}/model/pytorch/physics_losses.py')
            f = f'{dst}/model/pytorch/dcrnn_supervisor.py'
            s = open(f).read()
            old = 'from model.pytorch.loss import width,mis_loss,masked_mse_loss,masked_mae_loss,quantile_loss,underwood_loss'
            assert s.count(old) == 1
            s = s.replace(old, 'from model.pytorch.loss import width,mis_loss,masked_mse_loss,masked_mae_loss,quantile_loss\n'
                               'from model.pytorch.physics_losses import underwood_loss')
            open(f, 'w').write(s)
    BASES[name] = dst

prepare_base('gpde_quantile_phy_model_PEMSD8', 'pemsd8', '/content/data/PEMSD8', our_loss=True)
prepare_base('gpde_quantile_model', 'metrla', '/content/data/METR-LA', epochs=25)
prepare_base('gpde_quantile_phy_model', 'metrla_phy', '/content/data/METR-LA', epochs=25)

def make_run(base, name, seed, aligned=False):
    # Copy a base folder, set the seed and (optionally) the time-aligned physics loss.
    dst = f'{LOCAL}/{name}'
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(BASES[base], dst)
    os.makedirs(f'{dst}/models', exist_ok=True)
    f = f'{dst}/model/pytorch/dcrnn_supervisor.py'
    s = open(f).read()
    assert len(re.findall(r'^random_seed = 0\s*$', s, re.M)) == 1
    s = re.sub(r'^random_seed = 0\s*$', f'random_seed = {seed}', s, flags=re.M)
    if aligned:
        # keep the full labels (incl. future occupancy) and give them to the physics loss instead of the input
        a1 = "        x, y = self._get_x_y(x, y)\n"
        a2 = "reg* underwood_loss(y_pred, x_true, y_true)"
        assert s.count(a1) == 1 and s.count(a2) == 1
        s = s.replace(a1, a1 + "        self._y_full = y.to(device)  # labels incl. future occupancy\n")
        s = s.replace(a2, "reg* underwood_loss(y_pred, self._y_full if os.environ.get('PHYS_ALIGN') == 'future' else x_true, y_true)")
        s = 'import os\n' + s
    open(f, 'w').write(s)
    return name

def label(prefix, seed):
    return prefix if seed == 0 else f'{prefix}_seed{seed}'

ENV = {'underwood': {}, 'aligned': {'PHYS_ALIGN': 'future'},
       'greenshields': {'PHYS_LAW': 'greenshields', 'PHYS_VF': f'{PARAMS}/greenshields_v_f.npy',
                        'PHYS_K': f'{PARAMS}/greenshields_k_jam.npy'}}

def start(jobs):
    # jobs: list of (run name, extra env). Runs one after the other in the background,
    # copies checkpoints and logs to Drive every 5 minutes, and evaluates each run when it ends.
    cmds = [f"cd {LOCAL}/{n} && env -u PHYS_LAW -u PHYS_VF -u PHYS_K -u PHYS_ALIGN "
            f"{' '.join(f'{k}={v}' for k, v in e.items())} "
            f"python dcrnn_train_pytorch.py --config_filename=data/model/dcrnn_la.yaml > train.log 2>&1" for n, e in jobs]
    subprocess.Popen(['bash', '-c', '; '.join(cmds)], start_new_session=True)
    names = ' '.join(n for n, _ in jobs)
    backup = (f'while true; do sleep 300; for n in {names}; do mkdir -p "{WORK}/checkpoints/$n"; '
              f'cp -u {LOCAL}/$n/models/*.tar "{WORK}/checkpoints/$n/" 2>/dev/null; '
              f'cp {LOCAL}/$n/train.log "{WORK}/checkpoints/$n/train.log" 2>/dev/null; done; done')
    subprocess.Popen(['bash', '-c', backup], start_new_session=True)
    subprocess.Popen(['python', '/content/auto_eval.py'] + [n for n, _ in jobs], start_new_session=True)
    print('started:', [n for n, _ in jobs])

open('/content/auto_eval.py', 'w').write(f'''
import os, re, sys, time, glob, shutil, subprocess
LOCAL, WORK, IMPL, MS = {LOCAL!r}, {WORK!r}, {IMPL!r}, {METRICS_SCRIPTS!r}
def running(d):
    for p in glob.glob("/proc/[0-9]*"):
        try:
            if os.readlink(p + "/cwd") == d and "dcrnn_train" in open(p + "/cmdline").read(): return True
        except Exception: pass
    return False
for name in sys.argv[1:]:
    d, log = f"{{LOCAL}}/{{name}}", f"{{LOCAL}}/{{name}}/train.log"
    while not os.path.exists(log) or running(d): time.sleep(60)
    t = open(log).read()
    shutil.copy(log, f"{{WORK}}/logs/{{name}}.log")
    best = re.findall(r"saving to models/epo(\\d+)\\.tar", t)
    if not best: continue
    data = "/content/data/METR-LA" if name.startswith("metrla") else "/content/data/PEMSD8"
    r = subprocess.run(["python", f"{{IMPL}}/eval_run.py", "--run", d, "--epoch", best[-1], "--data_dir", data,
                        "--label", name, "--csv", f"{{WORK}}/results/horizon_metrics.csv", "--metrics_scripts", MS],
                       capture_output=True, text=True)
    open(f"{{WORK}}/results/eval_output_{{name}}.txt", "w").write(r.stdout + r.stderr)
''')
print('bases:', BASES)

Start the runs, one GPU per call (one Colab account or session each). For example:

In [ ]:
# PEMSD8, authors' law with our loss module, seed 0 (about 55 min on a T4)
start([(make_run('pemsd8', label('pemsd8_underwood_ours', 0), 0), ENV['underwood'])])

# Other runs, same pattern (uncomment one line per GPU session):
# start([(make_run('pemsd8', label('pemsd8_greenshields', 1), 1), ENV['greenshields'])])
# start([(make_run('pemsd8', label('pemsd8_underwood_aligned', 2), 2, aligned=True), ENV['aligned'])])
# start([(make_run('metrla', label('metrla_gpde_quantile', 1), 1), {})])
# start([(make_run('metrla_phy', label('metrla_gpde_quantile_phy', 1), 1), {})])

Progress of every run on this VM (epochs done, best checkpoint so far):

In [ ]:
for d in sorted(glob.glob(f'{LOCAL}/[!_]*')):
    log = f'{d}/train.log'
    if os.path.exists(log):
        t = open(log).read()
        best = re.findall(r'saving to models/epo(\d+)\.tar', t)
        print(f'{os.path.basename(d):34s} {t.count("val_mae"):3d} epochs   best epoch: {best[-1] if best else "-"}')
print(subprocess.run('ps aux | grep "python dcrnn_train" | grep -v grep | wc -l', shell=True,
                     capture_output=True, text=True).stdout.strip(), 'training process(es) running')

## 5. Evaluation per horizon

Each run is evaluated automatically when its training ends (rows appended to `results/horizon_metrics.csv`). If the session ended first, rebuild the run folder, put the best checkpoint from the Drive backup into `models/`, and evaluate by hand:

In [ ]:
def eval_by_hand(base, name, seed, epoch, aligned=False, env=None):
    make_run(base, name, seed, aligned)
    shutil.copy(f'{WORK}/checkpoints/{name}/epo{epoch}.tar', f'{LOCAL}/{name}/models/')
    data = '/content/data/METR-LA' if name.startswith('metrla') else '/content/data/PEMSD8'
    extra = []
    if env and env.get('PHYS_LAW') == 'greenshields':
        extra = ['--law', 'greenshields', '--vf', env['PHYS_VF'], '--k', env['PHYS_K']]
    r = subprocess.run(['python', f'{IMPL}/eval_run.py', '--run', f'{LOCAL}/{name}', '--epoch', str(epoch),
                        '--data_dir', data, '--label', name, '--csv', f'{WORK}/results/horizon_metrics.csv',
                        '--metrics_scripts', METRICS_SCRIPTS] + extra, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])

# example: eval_by_hand('pemsd8', 'pemsd8_underwood_aligned_seed1', 1, 48, aligned=True)

## 6. Summaries and figures

Test split, single-horizon view, paired by seed. Writes the summary CSVs in `results/` and the figures in `figures/`.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
HM = pd.read_csv(f'{WORK}/results/horizon_metrics.csv')
OUTR, OUTF = f'{WORK}/results', f'{WORK}/figures'
os.makedirs(OUTF, exist_ok=True)
M = ['mae', 'rmse', 'mis_authors', 'coverage', 'width']

def compare(groups, prefix, paired_name, title, colors=('tab:blue', 'tab:orange')):
    # groups: {variant name: run-label prefix}; seeds taken from the labels present in the CSV
    names = {}
    for var, pre in groups.items():
        for lab in HM['label'].unique():
            m = re.fullmatch(re.escape(pre) + r'(?:_seed(\d+))?', lab)
            if m:
                names[lab] = (var, int(m.group(1) or 0))
    t = HM[(HM['split'] == 'test') & HM['label'].isin(names)].copy()
    t['variant'] = t['label'].map(lambda l: names[l][0]); t['seed'] = t['label'].map(lambda l: names[l][1])
    t.to_csv(f'{OUTR}/{prefix}.csv', index=False)
    s = t[t['view'] == 'step']
    s.groupby(['variant', 'minutes'])[M].agg(['mean', 'std', 'min', 'max']).round(4).to_csv(f'{OUTR}/{prefix}_summary.csv')
    a, b = list(groups)
    p = s.pivot_table(index=['seed', 'minutes'], columns='variant', values=M)
    pd.concat({k: p[k][b] - p[k][a] for k in M}, axis=1).round(4).to_csv(f'{OUTR}/{paired_name}.csv')
    print(s.groupby(['minutes', 'variant'])[M].agg(['mean', 'std']).round(3).to_string())
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.8)); H = [15, 30, 60]
    for x, (k, lab) in zip(ax, [('mae', 'MAE (mph)'), ('mis_authors', 'MIS (authors)'), ('coverage', 'Coverage of 95% interval')]):
        for var, col, off in [(a, colors[0], -0.12), (b, colors[1], 0.12)]:
            g = s[s['variant'] == var]
            for i, h in enumerate(H):
                v = g[g['minutes'] == h][k]
                x.scatter([i + off] * len(v), v, color=col, alpha=0.45, s=18)
                x.errorbar(i + off, v.mean(), yerr=v.std(), fmt='o', color=col, capsize=4, ms=7,
                           label=f'{var} (mean ± std)' if i == 0 else None)
        x.set_xticks(range(3)); x.set_xticklabels(H); x.set_xlabel('Horizon (min)'); x.set_ylabel(lab)
    ax[2].axhline(0.95, ls='--', color='grey', lw=1); ax[0].legend(fontsize=8)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(f'{OUTF}/{prefix}.png', dpi=150); plt.show()

compare({'Underwood': 'pemsd8_underwood_ours', 'Greenshields': 'pemsd8_greenshields'},
        'greenshields_vs_underwood', 'greenshields_minus_underwood_paired', 'PEMSD8: Underwood vs Greenshields physics loss (test)')
compare({"Underwood (authors')": 'pemsd8_underwood_ours', 'Time-aligned': 'pemsd8_underwood_aligned'},
        'time_aligned_vs_underwood', 'time_aligned_minus_underwood_paired', "PEMSD8: authors' 60-min-lagged occupancy vs time-aligned (test)",
        colors=('tab:blue', 'tab:green'))
compare({'without physics': 'metrla_gpde_quantile', 'with physics': 'metrla_gpde_quantile_phy'},
        'metrla_tod', 'metrla_phy_minus_nophy_paired', 'METR-LA: GPDE+Quantile without vs with the physics loss (test)')

## 7. Calibration check of the time-aligned loss

Saves validation and test predictions of the six PEMSD8 runs (authors' loss and time-aligned, 3 seeds each), then runs the team's `extensions/coverage_conformal.py` (raw, CQR, per-sensor CQR, ACI; validation split as calibration set). Uses the checkpoint backups on Drive, so it works on a CPU runtime after training.

In [ ]:
SIX = {'underwood_s0': ('pemsd8_underwood_ours', 0, False), 'underwood_s1': ('pemsd8_underwood_ours_seed1', 1, False),
       'underwood_s2': ('pemsd8_underwood_ours_seed2', 2, False), 'aligned_s0': ('pemsd8_underwood_aligned', 0, True),
       'aligned_s1': ('pemsd8_underwood_aligned_seed1', 1, True), 'aligned_s2': ('pemsd8_underwood_aligned_seed2', 2, True)}
os.makedirs('/content/conformal_npz', exist_ok=True)
for short, (name, seed, aligned) in SIX.items():
    best = re.findall(r'saving to models/epo(\d+)\.tar', open(f'{WORK}/logs/{name}.log').read())[-1]
    make_run('pemsd8', name, seed, aligned)
    shutil.copy(f'{WORK}/checkpoints/{name}/epo{best}.tar', f'{LOCAL}/{name}/models/')
    subprocess.run(['python', f'{IMPL}/eval_run.py', '--run', f'{LOCAL}/{name}', '--epoch', best,
                    '--data_dir', '/content/data/PEMSD8', '--label', short, '--csv', '/content/conformal_tmp.csv',
                    '--metrics_scripts', METRICS_SCRIPTS, '--save_npz', f'/content/conformal_npz/{short}.npz'], check=True)
    print(short, 'epoch', best)

args = ['python', f'{REPO}/extensions/coverage_conformal.py', '--out', f'{WORK}/conformal']
for short in SIX:
    args += ['--npz', f'{short}=/content/conformal_npz/{short}.npz']
subprocess.run(args, check=True)

c = pd.read_csv(f'{WORK}/conformal/coverage_conformal_metrics.csv')
c['variant'] = c['model'].str.split('_').str[0]
cols = ['MIS', 'width', 'coverage', 'cov[congested <40 mph]']
print(c[c['horizon'] == 'avg_1-12'].groupby(['method', 'variant'])[cols].mean().round(3).to_string())